# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [3]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cosmopedia_id.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

file_size_mb = DATA_PATH.stat().st_size / (1024 * 1024)

print(f"File: {DATA_PATH.name}")
print(f"Ukuran: {file_size_mb:.2f} MB")

File: cosmopedia_id.csv
Ukuran: 554.81 MB


## Profil Polars

Gunakan `scan_csv` untuk membaca dataset secara lazy sehingga proses profiling lebih efisien pada dataset berukuran besar.

In [4]:
lf = pl.scan_csv(DATA_PATH)

print("Schema Dataset:")
print(lf.collect_schema())

print("\nJumlah Baris:")
print(lf.select(pl.len()).collect())

print("\nJumlah Missing Values:")
print(
    lf.select([
        pl.col(column).null_count().alias(column)
        for column in lf.collect_schema().names()
    ]).collect()
)

Schema Dataset:
Schema({'id': Int64, 'prompt': String, 'text': String, 'token_length': Int64, 'audience': String, 'format': String, 'seed_data': String, 'datasheet_global_id': Int64, 'created_at': String})

Jumlah Baris:
shape: (1, 1)
┌────────┐
│ len    │
│ ---    │
│ u32    │
╞════════╡
│ 120068 │
└────────┘

Jumlah Missing Values:
shape: (1, 9)
┌─────┬────────┬──────┬──────────────┬───┬────────┬───────────┬─────────────────────┬────────────┐
│ id  ┆ prompt ┆ text ┆ token_length ┆ … ┆ format ┆ seed_data ┆ datasheet_global_id ┆ created_at │
│ --- ┆ ---    ┆ ---  ┆ ---          ┆   ┆ ---    ┆ ---       ┆ ---                 ┆ ---        │
│ u32 ┆ u32    ┆ u32  ┆ u32          ┆   ┆ u32    ┆ u32       ┆ u32                 ┆ u32        │
╞═════╪════════╪══════╪══════════════╪═══╪════════╪═══════════╪═════════════════════╪════════════╡
│ 0   ┆ 0      ┆ 0    ┆ 0            ┆ … ┆ 0      ┆ 0         ┆ 0                   ┆ 0          │
└─────┴────────┴──────┴──────────────┴───┴────────┴─────

In [5]:
lf.head(10).collect()

id,prompt,text,token_length,audience,format,seed_data,datasheet_global_id,created_at
i64,str,str,i64,str,str,str,i64,str
1,"""Berikut adalah ekstrak dari ha…","""Dahulu kala, ada sebuah agen y…",699,"""children""","""story""","""fineweb""",1,"""2026-02-18 07:35:54"""
24,"""Berikut kutipan dari sebuah ha…","""Suatu ketika, di sebuah negeri…",137,"""children""","""story""","""fineweb""",2,"""2026-02-18 08:19:23"""
25,"""Berikut kutipan dari sebuah ha…","""Di kota Popville, hiduplah ban…",93,"""children""","""story""","""fineweb""",3,"""2026-02-18 08:19:23"""
26,"""Berikut kutipan tentang chipmu…","""Di sebuah lingkungan yang tena…",76,"""children""","""story""","""fineweb""",4,"""2026-02-18 08:19:23"""
27,"""Berikut kutipan tentang terapi…","""Timmy pernah merasa sangat sed…",88,"""children""","""story""","""fineweb""",5,"""2026-02-18 08:19:23"""
28,"""Berikut kutipan tentang invest…","""Di negeri yang luas, seorang b…",57,"""children""","""story""","""fineweb""",6,"""2026-02-18 08:19:23"""
29,"""Berikut kutipan tentang kesela…","""Alice: Hei Bob, kalau jalan di…",61,"""children""","""dialogue""","""fineweb""",7,"""2026-02-18 08:19:23"""
30,"""Berikut kutipan tentang makana…","""Emma suka makanan manis. Bu Gu…",64,"""children""","""story""","""fineweb""",8,"""2026-02-18 08:19:23"""
31,"""Berikut kutipan tentang penyem…","""Alice: Aku sering susah tidur …",48,"""children""","""dialogue""","""fineweb""",9,"""2026-02-18 08:19:23"""


## Profil DuckDB

Gunakan DuckDB untuk melakukan profiling statistik dataset secara langsung dari file CSV menggunakan fungsi `SUMMARIZE`.

In [6]:
con = duckdb.connect()

profile = con.execute(
    "SUMMARIZE SELECT * FROM read_csv_auto(?)",
    [str(DATA_PATH)]
).fetchdf()

display(profile)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,id,BIGINT,1,120091,117463,60057.49921711031,34660.791753253696,29817,59986,89858,120068,0.0
1,prompt,VARCHAR,"Berikut adalah cuplikan dari halaman web: ""Buk...",Write an educational story (3-5 paragraphs) ta...,114146,NaN,NaN,NaN,NaN,NaN,120068,0.0
2,text,VARCHAR,**1.3 Combinational Circuits**\n\nWelcome bac...,अगर आपको उchcha madhyamik standard mein हो तो ...,115976,NaN,NaN,NaN,NaN,NaN,120068,0.0
3,token_length,BIGINT,36,2787,1545,1187.9396508645102,194.61858027220896,1065,1168,1297,120068,0.0
4,audience,VARCHAR,children,young_children,3,NaN,NaN,NaN,NaN,NaN,120068,0.0
5,format,VARCHAR,Dialogue,textbook_unconditionned_topic,8,NaN,NaN,NaN,NaN,NaN,120068,0.0
6,seed_data,VARCHAR,fineweb,ultrachat,5,NaN,NaN,NaN,NaN,NaN,120068,0.0
7,datasheet_global_id,BIGINT,1,120068,117463,60034.5,34660.79039779674,29794,59963,89835,120068,0.0
8,created_at,TIMESTAMP,2026-02-18 07:35:54,2026-03-06 03:44:48,123551,2026-02-27 01:14:04.863586,NaN,2026-02-24 19:26:21.490752,2026-02-27 10:49:54.012115,2026-03-01 19:08:02.55214,120068,0.0


## Catatan Milestone 1

Berdasarkan hasil profiling, dataset Cosmopedia Indonesia memiliki ukuran sekitar **554,81 MB**, dengan **120.068 baris dan 9 kolom**. Tipe data terdiri dari kolom numerik (`id`, `token_length`, dan `datasheet_global_id`) serta kolom teks (`prompt`, `text`, `audience`, `format`, `seed_data`, dan `created_at`).

Ditemukan masing-masing satu nilai kosong pada kolom `audience`, `format`, dan `seed_data`. Dengan demikian, ketiga kolom tersebut memiliki jumlah missing values terbanyak. Tidak ditemukan nilai kosong pada kolom utama `id`, `prompt`, dan `text`.

Anomali lain yang perlu diperhatikan adalah adanya potensi outlier pada kolom `token_length`, yaitu sebanyak **3.790 data (3,16%)** berdasarkan metode IQR. Tidak ditemukan baris duplikat maupun ID duplikat. Temuan ini menjadi dasar penentuan strategi pembersihan data pada Milestone 2.